# Case Study 02 — PD Model: Logistic Panel Regression

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Methodology

**Model:** Logistic regression with institution fixed effects + time dummies  
**Target:** `y_deterioration` — binary IFRS9 Stage1→2 proxy (NB02)  
**Regularization:** L2 Ridge — controls overfitting on short panels  

| Section | Content |
|---------|---------|
| 1 | Load panel + metadata contract |
| 2 | VIF-based feature selection |
| 3 | Temporal train/test split (no leakage) |
| 4 | Institution FE + time dummies encoding |
| 5 | L2 regularization sweep |
| 6 | Final model: coefficients + odds ratios |
| 7 | SHAP feature importance |
| 8 | Export model artifacts |


In [ ]:
import warnings, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

BASE   = Path('../')
PANEL  = BASE / 'data' / 'panel'
FIGS   = BASE / 'reports' / 'figures'
MODELS = BASE / 'models'
for p in [FIGS, MODELS]: p.mkdir(parents=True, exist_ok=True)

TEAL = '#01696f'; MAROON = '#a12c7b'; GRAY = '#bab9b4'
GREEN = '#437a22'; ORANGE = '#964219'; GOLD = '#d19900'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#dcd9d5', 'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974', 'ytick.color': '#7a7974',
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'semibold', 'figure.dpi': 130,
})
print('Environment ready.')

## 1. Load Panel

In [ ]:
df = pd.read_parquet(PANEL / 'panel_model_ready.parquet')
with open(PANEL / 'panel_metadata.json') as f:
    meta = json.load(f)

df['period'] = pd.to_datetime(df['period'])
ALL_FEATURES = meta['all_features']
TARGET = meta['target']

print(f'Panel: {len(df):,} rows | {df.institution_id.nunique()} institutions')
print(f'Target rate: {df[TARGET].mean():.2%}')
print(f'Features: {len(ALL_FEATURES)}')

## 2. VIF-Based Feature Selection

In [ ]:
candidate_feats = [f for f in ALL_FEATURES if f in df.columns]
X_vif = df[candidate_feats].dropna()

vif_data = pd.DataFrame({
    'feature': candidate_feats,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(len(candidate_feats))]
}).sort_values('VIF', ascending=False)

high_vif = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
FINAL_FEATURES = [f for f in candidate_feats if f not in high_vif]
if len(FINAL_FEATURES) < 3:
    FINAL_FEATURES = candidate_feats[:5]

display(vif_data.round(2))
print(f'Dropped (VIF>10): {high_vif}')
print(f'Final features  : {FINAL_FEATURES}')

## 3. Temporal Train/Test Split

Random split causes data leakage in panel time series.  
Train = first 80% of sorted periods | Test = last 20%.

In [ ]:
sorted_periods = sorted(df['period'].unique())
cutoff_idx  = int(len(sorted_periods) * 0.80)
cutoff_date = sorted_periods[cutoff_idx]

train = df[df['period'] < cutoff_date].copy()
test  = df[df['period'] >= cutoff_date].copy()

X_train = train[FINAL_FEATURES]
y_train = train[TARGET]
X_test  = test[FINAL_FEATURES]
y_test  = test[TARGET]

print(f'Cutoff : {cutoff_date.date()}')
print(f'Train  : {len(train):,} rows | target rate: {y_train.mean():.2%}')
print(f'Test   : {len(test):,} rows  | target rate: {y_test.mean():.2%}')

## 4. Institution Fixed Effects + Time Dummies

In [ ]:
fe_train = pd.get_dummies(train['institution_id'], prefix='fe', drop_first=True).astype(int)
fe_test  = pd.get_dummies(test['institution_id'],  prefix='fe', drop_first=True).astype(int)
fe_test  = fe_test.reindex(columns=fe_train.columns, fill_value=0)

train['time_dummy'] = train['period'].dt.to_period('Q').astype(str)
test['time_dummy']  = test['period'].dt.to_period('Q').astype(str)
td_train = pd.get_dummies(train['time_dummy'], prefix='td', drop_first=True).astype(int)
td_test  = pd.get_dummies(test['time_dummy'],  prefix='td', drop_first=True).astype(int)
td_test  = td_test.reindex(columns=td_train.columns, fill_value=0)

X_train_fe = pd.concat([X_train.reset_index(drop=True),
                         fe_train.reset_index(drop=True),
                         td_train.reset_index(drop=True)], axis=1)
X_test_fe  = pd.concat([X_test.reset_index(drop=True),
                         fe_test.reset_index(drop=True),
                         td_test.reset_index(drop=True)], axis=1)

print(f'Feature matrix: {X_train_fe.shape}')
print(f'  Economic : {len(FINAL_FEATURES)} | FE dummies: {fe_train.shape[1]} | Time dummies: {td_train.shape[1]}')

## 5. L2 Regularization Sweep

In [ ]:
C_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
n_splits = min(5, max(2, len(sorted_periods) // 4))
skf = StratifiedKFold(n_splits=n_splits, shuffle=False)

scaler_sw = StandardScaler()
X_sw = scaler_sw.fit_transform(X_train_fe.fillna(0))

cv_results = []
for C in C_grid:
    m = LogisticRegression(C=C, penalty='l2', solver='lbfgs',
                           max_iter=2000, class_weight='balanced')
    scores = cross_val_score(m, X_sw, y_train, cv=skf, scoring='roc_auc')
    cv_results.append({'C': C, 'mean_AUC': scores.mean(), 'std_AUC': scores.std()})

cv_df  = pd.DataFrame(cv_results)
best_C = float(cv_df.loc[cv_df['mean_AUC'].idxmax(), 'C'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cv_df['C'], cv_df['mean_AUC'], color=TEAL, lw=2, marker='o', markersize=5)
ax.fill_between(cv_df['C'],
                cv_df['mean_AUC'] - cv_df['std_AUC'],
                cv_df['mean_AUC'] + cv_df['std_AUC'],
                alpha=0.12, color=TEAL)
ax.axvline(best_C, color=ORANGE, ls='--', lw=1.5, label=f'Best C={best_C}')
ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization)'); ax.set_ylabel('CV AUC')
ax.set_title('L2 Regularization Sweep — CV AUC')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGS / 'model_01_reg_sweep.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Best C = {best_C}')

## 6. Final Model: Coefficients + Odds Ratios

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_fe.fillna(0))
X_test_sc  = scaler.transform(X_test_fe.fillna(0))

model = LogisticRegression(C=best_C, penalty='l2', solver='lbfgs',
                           max_iter=2000, class_weight='balanced')
model.fit(X_train_sc, y_train)

y_prob_train = model.predict_proba(X_train_sc)[:, 1]
y_prob_test  = model.predict_proba(X_test_sc)[:, 1]
auc_train    = roc_auc_score(y_train, y_prob_train)
auc_test     = roc_auc_score(y_test,  y_prob_test)

print(f'AUC Train : {auc_train:.4f}')
print(f'AUC Test  : {auc_test:.4f}')
gap = auc_train - auc_test
print(f'Overfit gap: {gap:.4f}', '⚠ consider stronger L2' if gap > 0.08 else '✓ acceptable')

In [ ]:
feat_names = list(X_train_fe.columns)
econ_idx   = [i for i, f in enumerate(feat_names) if not f.startswith(('fe_', 'td_'))]

coef_df = pd.DataFrame({
    'feature'   : [feat_names[i] for i in econ_idx],
    'coef'      : model.coef_[0][econ_idx].round(4),
    'odds_ratio': np.exp(model.coef_[0][econ_idx]).round(4),
}).sort_values('coef', key=abs, ascending=False)
coef_df['direction'] = coef_df['coef'].apply(lambda x: '↑ risk' if x > 0 else '↓ risk')

fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(coef_df)*0.5)))

# Odds ratio chart
colors_or = [MAROON if od > 1 else TEAL for od in coef_df['odds_ratio']]
axes[0].barh(coef_df['feature'], coef_df['odds_ratio'] - 1, color=colors_or, alpha=0.8)
axes[0].axvline(0, color=GRAY, lw=0.9)
axes[0].set_xlabel('Odds Ratio − 1')
axes[0].set_title('Odds Ratios (standardized features)')
for i, (_, row) in enumerate(coef_df.iterrows()):
    axes[0].text(row['odds_ratio']-1+0.01, i, f'{row["odds_ratio"]:.3f}',
                va='center', fontsize=8)

# ROC curve
fpr_tr, tpr_tr, _ = roc_curve(y_train, y_prob_train)
fpr_te, tpr_te, _ = roc_curve(y_test,  y_prob_test)
axes[1].plot(fpr_tr, tpr_tr, color=TEAL,   lw=2, label=f'Train AUC={auc_train:.3f}')
axes[1].plot(fpr_te, tpr_te, color=ORANGE, lw=2, label=f'Test  AUC={auc_test:.3f}')
axes[1].plot([0,1],[0,1], color=GRAY, lw=1, ls='--', label='Random')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — PD Logit Panel')
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'model_02_coef_roc.png', bbox_inches='tight', dpi=150)
plt.show()
display(coef_df)

## 7. SHAP Feature Importance

In [ ]:
try:
    import shap
    explainer   = shap.LinearExplainer(model, X_train_sc, feature_names=feat_names)
    shap_values = explainer.shap_values(X_test_sc[:min(200, len(X_test_sc))])
    shap_econ   = shap_values[:, econ_idx]
    econ_names  = [feat_names[i] for i in econ_idx]
    shap_mean   = np.abs(shap_econ).mean(axis=0)
    shap_df = pd.DataFrame({'feature': econ_names, 'mean_abs_shap': shap_mean})
    shap_df = shap_df.sort_values('mean_abs_shap', ascending=True)
    fig, ax = plt.subplots(figsize=(7, max(4, len(shap_df)*0.45)))
    ax.barh(shap_df['feature'], shap_df['mean_abs_shap'], color=TEAL, alpha=0.8)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('SHAP Feature Importance (economic features)')
    plt.tight_layout()
    plt.savefig(FIGS / 'model_03_shap.png', bbox_inches='tight', dpi=150)
    plt.show()
except ImportError:
    print('Install shap: pip install shap')

## 8. Export Model Artifacts

In [ ]:
pipeline_obj = {
    'scaler': scaler, 'model': model,
    'features': FINAL_FEATURES,
    'fe_cols': list(fe_train.columns),
    'td_cols': list(td_train.columns),
}
joblib.dump(pipeline_obj, MODELS / 'pd_logit_panel.pkl')

y_pred_test = (y_prob_test >= 0.5).astype(int)
perf = {
    'created': str(datetime.date.today()),
    'model': 'LogisticRegression L2 + institution FE + time dummies',
    'best_C': best_C,
    'n_features_economic': len(FINAL_FEATURES),
    'n_features_total': int(X_train_fe.shape[1]),
    'train_auc': round(auc_train, 4),
    'test_auc': round(auc_test, 4),
    'overfit_gap': round(float(auc_train - auc_test), 4),
    'test_precision': round(float(precision_score(y_test, y_pred_test, zero_division=0)), 4),
    'test_recall': round(float(recall_score(y_test, y_pred_test, zero_division=0)), 4),
    'test_f1': round(float(f1_score(y_test, y_pred_test, zero_division=0)), 4),
    'cutoff_date': str(cutoff_date.date()),
    'economic_features': FINAL_FEATURES,
}
with open(MODELS / 'model_performance.json', 'w') as f:
    json.dump(perf, f, indent=2)

print('[saved] models/pd_logit_panel.pkl')
print('[saved] models/model_performance.json')
print()
print('=== MODEL PERFORMANCE SUMMARY ===')
for k, v in perf.items():
    if k != 'economic_features':
        print(f'  {k:<32} {v}')
print(f'  {"economic_features":<32} {FINAL_FEATURES}')
print()
print('Next → 05_validation.ipynb')